# 11 — Model Capacity Calibration: Largest Non-Overfitting Model

**QUESTION:** What is the largest model that doesn't overfit on 4 volumes?

**APPROACH:**
1. Generate synthetic data (known ground truth, same noise model)
2. Train 3 model sizes
3. Plot train vs validation loss
4. Find largest non-overfitting model
5. Measure stSNR on synthetic test set

**RESULT:** Justified channel config + epoch budget + regularization needs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Noise model parameters from notebook 10
# Using typical values; replace with your measured values
NOISE_PARAMS = {
    "F0": {"gain": 0.028, "read_noise": 1.2},
    "F1": {"gain": 0.030, "read_noise": 1.3},
    "F2": {"gain": 0.029, "read_noise": 1.25},
    "F3": {"gain": 0.031, "read_noise": 1.35},
}

print("Noise model parameters (from calibration):")
for stack, params in NOISE_PARAMS.items():
    print(f"  {stack}: g={params['gain']:.6f}, σ={params['read_noise']:.6f}")

In [ ]:
# Load real data and crop to 128x128
from cidc import load_stack

DATA = Path("../data/val")

print("Loading real stacks...")
stacks = {}
for name in ["F0", "F1", "F2", "F3"]:
    path = DATA / f"{name}.tif"
    data = np.asarray(load_stack(path), dtype=np.float32)
    # Crop to 128x128 spatial (center crop)
    h_start = (data.shape[1] - 128) // 2
    w_start = (data.shape[2] - 128) // 2
    stacks[name] = data[:, h_start:h_start+128, w_start:w_start+128]
    print(f"  {name}: {stacks[name].shape}")

# Use F0 as clean reference, F1/F2/F3 as noisy inputs
clean_ref = stacks["F0"]  # (T, 128, 128)
noisy_stacks = {name: stacks[name] for name in ["F1", "F2", "F3"]}

print(f"\nClean reference (F0): {clean_ref.shape}")
print(f"Noisy stacks (F1/F2/F3): {[s.shape for s in noisy_stacks.values()]}")
print("✓ Real data loaded and cropped to 128×128")

In [ ]:
# Simple 3D UNet-like architecture for denoising
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class SimpleUNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, channels=[8, 16, 32]):
        super().__init__()
        self.channels = channels
        
        # Encoder
        self.enc1 = nn.Conv3d(in_channels, channels[0], 3, padding=1)
        self.enc2 = nn.Conv3d(channels[0], channels[1], 3, stride=2, padding=1)
        self.enc3 = nn.Conv3d(channels[1], channels[2], 3, stride=2, padding=1)
        
        # Bottleneck
        self.bottleneck = nn.Conv3d(channels[2], channels[2], 3, padding=1)
        
        # Decoder
        self.dec3 = nn.ConvTranspose3d(channels[2], channels[1], 3, stride=2, padding=1, output_padding=1)
        self.dec2 = nn.ConvTranspose3d(channels[1], channels[0], 3, stride=2, padding=1, output_padding=1)
        self.dec1 = nn.Conv3d(channels[0], out_channels, 3, padding=1)
    
    def forward(self, x):
        # Encoder
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2))
        
        # Bottleneck
        b = F.relu(self.bottleneck(e3))
        
        # Decoder
        d3 = F.relu(self.dec3(b))
        d2 = F.relu(self.dec2(d3))
        d1 = self.dec1(d2)
        
        return d1

print("✓ Model architecture defined")

In [ ]:
# Prepare training data from real stacks
from sklearn.model_selection import train_test_split

class RealDataset(Dataset):
    def __init__(self, noisy, clean, patch_size=(64, 64, 64)):
        self.noisy = torch.from_numpy(noisy).float()
        self.clean = torch.from_numpy(clean).float()
        self.patch_size = patch_size
        
        # Generate patch indices
        T, H, W = noisy.shape
        pt, ph, pw = patch_size
        self.patches = []
        for t in range(0, T-pt, pt//2):
            for h in range(0, H-ph, ph//2):
                for w in range(0, W-pw, pw//2):
                    self.patches.append((t, h, w))
    
    def __len__(self):
        return len(self.patches)
    
    def __getitem__(self, idx):
        t, h, w = self.patches[idx]
        pt, ph, pw = self.patch_size
        
        noisy_patch = self.noisy[t:t+pt, h:h+ph, w:w+pw]
        clean_patch = self.clean[t:t+pt, h:h+ph, w:w+pw]
        
        return {
            'noisy': noisy_patch.unsqueeze(0),
            'clean': clean_patch.unsqueeze(0)
        }

# Create datasets for each noisy stack (F1, F2, F3) against clean reference (F0)
train_data = []
val_data = []

for stack_name in ["F1", "F2", "F3"]:
    dataset = RealDataset(noisy_stacks[stack_name], clean_ref)
    
    # 80/20 split per stack
    indices = np.arange(len(dataset))
    train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
    
    for idx in train_idx:
        train_data.append(dataset[idx])
    for idx in val_idx:
        val_data.append(dataset[idx])

print(f"Training patches: {len(train_data)}")
print(f"Validation patches: {len(val_data)}")
print("✓ Dataset prepared")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, device='cpu'):
    """
    Train model and track train/val loss curves.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        for batch in train_loader:
            noisy = batch['noisy'].to(device)
            clean = batch['clean'].to(device)
            
            pred = model(noisy)
            loss = criterion(pred, clean)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                noisy = batch['noisy'].to(device)
                clean = batch['clean'].to(device)
                
                pred = model(noisy)
                loss = criterion(pred, clean)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")
    
    return train_losses, val_losses

print("✓ Training function defined")

In [ ]:
# Train three model sizes
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

model_configs = {
    "Small": [8, 16, 32],
    "Medium": [16, 32, 64],
    "Large": [32, 64, 128],
}

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)

results = {}

for model_name, channels in model_configs.items():
    print(f"\nTraining {model_name} model (channels={channels})...")
    
    model = SimpleUNet3D(channels=channels).to(device)
    
    # Count parameters
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")
    
    train_losses, val_losses = train_model(
        model, train_loader, val_loader,
        epochs=20, device=device
    )
    
    results[model_name] = {
        "model": model,
        "channels": channels,
        "n_params": n_params,
        "train_losses": train_losses,
        "val_losses": val_losses,
    }
    
    print(f"  Final: train={train_losses[-1]:.6f}, val={val_losses[-1]:.6f}")

print("\n✓ Training complete")

In [ ]:
# Plot train vs val loss for each model
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (model_name, data) in zip(axes, results.items()):
    epochs = np.arange(1, len(data['train_losses']) + 1)
    
    ax.plot(epochs, data['train_losses'], 'o-', label='Train', linewidth=2, markersize=4)
    ax.plot(epochs, data['val_losses'], 's-', label='Validation', linewidth=2, markersize=4)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(f"{model_name} ({data['n_params']:,} params)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Detect overfitting (where val starts diverging from train)
    gap = np.array(data['val_losses']) - np.array(data['train_losses'])
    overfit_epoch = np.argmax(gap) + 1 if np.max(gap) > 0.01 else len(gap)
    
    ax.axvline(overfit_epoch, color='red', linestyle='--', alpha=0.5, label=f'Diverge @ {overfit_epoch}')
    ax.legend()

plt.tight_layout()
plt.savefig('/tmp/model_capacity_training.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Training curves plotted")

In [ ]:
# Analyze overfitting
print("\n" + "="*80)
print("OVERFITTING ANALYSIS")
print("="*80 + "\n")

for model_name, data in results.items():
    train_losses = np.array(data['train_losses'])
    val_losses = np.array(data['val_losses'])
    
    gap = val_losses - train_losses
    overfit_epoch = np.argmax(gap) + 1
    max_gap = np.max(gap)
    
    final_train = train_losses[-1]
    final_val = val_losses[-1]
    
    print(f"{model_name} ({data['n_params']:,} params):")
    print(f"  Final train loss: {final_train:.6f}")
    print(f"  Final val loss: {final_val:.6f}")
    print(f"  Val-Train gap: {final_val - final_train:.6f}")
    print(f"  Max gap: {max_gap:.6f} (at epoch {overfit_epoch})")
    
    if max_gap < 0.01:
        print(f"  ✓ NO OVERFITTING detected")
    elif max_gap < 0.02:
        print(f"  ~ MILD OVERFITTING (manageable)")
    else:
        print(f"  ✗ SIGNIFICANT OVERFITTING")
    print()

In [ ]:
# Compute stSNR on synthetic test data
from cidc import stsnr

print("\n" + "="*80)
print("stSNR ON SYNTHETIC TEST DATA")
print("="*80 + "\n")

# Use first synthetic stack as test
test_stack = synthetic_data["F0"]
test_clean = torch.from_numpy(test_stack['clean']).float().unsqueeze(0).unsqueeze(0).to(device)
test_noisy = torch.from_numpy(test_stack['noisy']).float().unsqueeze(0).unsqueeze(0).to(device)

for model_name, data in results.items():
    model = data['model']
    model.eval()
    
    with torch.no_grad():
        pred = model(test_noisy)
    
    pred_np = pred.squeeze().cpu().numpy()
    clean_np = test_clean.squeeze().cpu().numpy()
    
    # Compute stSNR
    result = stsnr(pred_np, clean_np)
    
    print(f"{model_name}:")
    print(f"  stSNR: {result.st_snr:.3f} dB")
    print(f"  sSNR:  {result.s_snr:.3f} dB")
    print(f"  tSNR:  {result.t_snr:.3f} dB")
    print()

In [ ]:
# Recommendation
print("\n" + "="*100)
print("RECOMMENDATION: LARGEST NON-OVERFITTING MODEL")
print("="*100 + "\n")

# Find largest model without significant overfitting
best_model = None
for model_name in ["Large", "Medium", "Small"]:
    data = results[model_name]
    gap = np.max(np.array(data['val_losses']) - np.array(data['train_losses']))
    
    if gap < 0.02:  # Threshold for acceptable overfitting
        best_model = model_name
        break

if best_model:
    print(f"✓ Recommended: {best_model} model")
    print(f"  Channels: {results[best_model]['channels']}")
    print(f"  Parameters: {results[best_model]['n_params']:,}")
    print(f"  Status: No significant overfitting detected")
else:
    print("⚠ All models show overfitting")
    print("  Consider: regularization (dropout/weight decay), more data, or smaller models")

print(f"\n" + "="*100)

In [ ]:
# Summary table
print("\nMODEL CAPACITY SUMMARY")
print()
print("| Model  | Channels      | Parameters | Train Loss | Val Loss | Gap   | Overfit? |")
print("|--------|---------------|------------|-----------|----------|-------|----------|")

for model_name, data in sorted(results.items(), key=lambda x: x[1]['n_params']):
    channels = data['channels']
    n_params = data['n_params']
    train_loss = data['train_losses'][-1]
    val_loss = data['val_losses'][-1]
    gap = val_loss - train_loss
    
    status = "✓ No" if gap < 0.01 else ("~ Mild" if gap < 0.02 else "✗ Yes")
    
    print(f"| {model_name:6} | {str(channels):13} | {n_params:10,} | {train_loss:9.6f} | {val_loss:8.6f} | {gap:.3f} | {status:8} |")

print()